In [1]:
import pandas as pd

In [3]:
# 1. Define the hierarchy
columns = [
    ('Units', ''), 
    ('Breakfast', 'Items Sold'), ('Breakfast', 'Sales'),
    ('Lunch', 'Items Sold'), ('Lunch', 'Sales'),
    ('Dinner', 'Items Sold'), ('Dinner', 'Sales'),
    ('Total Items Sold', ''),
    ('Total Sales', '')
]
multi_col = pd.MultiIndex.from_tuples(columns)

# 2. Prepare your data as a list of lists
data_rows = [
    ['Store 1', 10, 50, 20, 100, 30, 150, 60, 300],
    ['Store 2', 15, 75, 25, 125, 35, 175, 75, 375]
]

# 3. Create the DataFrame in one shot
df = pd.DataFrame(data_rows, columns=multi_col)

print(df)

     Units  Breakfast            Lunch           Dinner        \
           Items Sold Sales Items Sold Sales Items Sold Sales   
0  Store 1         10    50         20   100         30   150   
1  Store 2         15    75         25   125         35   175   

  Total Items Sold Total Sales  
                                
0               60         300  
1               75         375  


In [4]:
df

Units  Breakfast            Lunch           Dinner        \
           Items Sold Sales Items Sold Sales Items Sold Sales   
0  Store 1         10    50         20   100         30   150   
1  Store 2         15    75         25   125         35   175   

  Total Items Sold Total Sales  
                                
0               60         300  
1               75         375

In [6]:
dff = pd.read_excel('C:/Users/mprasanna/Desktop/misc/patron_count_raw.xlsx', sheet_name='RawData')
dff

,theader_id,tdatetime,location_name,theader_type,Checks,item_price,kiosk_name,tdetail_id,item_qty,item_number,item_description,item_cat_name,item_sub_cat_name,item_split,item_split_percent,ttenders_id,ttenders_tid,ttenders_name,Meal Period,Unit
0,15624070,2026-04-09 10:05:22,RTCC Food Court,1,1,13.99,RTCC C and G MO,19176683,1,CG006,PB Chocolate Acai Medium,C&G,C&G Meal Plan,False,0,15226235,15624070,Meal Exchange,Breakfast,RTCC_C&G
1,15624071,2026-04-09 10:05:22,RTCC Food Court,1,1,11.99,RTCC C and G MO,19176684,1,CG011,Protein Power Medium,C&G,C&G Meal Plan,False,0,15226236,15624071,Meal Exchange,Breakfast,RTCC_C&G
2,15624042,2026-04-09 10:03:27,RTCC Food Court,1,1,11.99,RTCC C and G MO,19176654,1,CG011,Protein Power Medium,C&G,C&G Meal Plan,False,0,15226208,15624042,Meal Exchange,Breakfast,RTCC_C&G
3,15624039,2026-04-09 10:03:24,RTCC Food Court,1,1,13.99,RTCC C and G MO,19176650,1,CG004,Trojan Acai Medium,C&G,C&G Meal Plan,False,0,15226205,15624039,Meal Exchange,Breakfast,RTCC_C&G
4,15624038,2026-04-09 10:03:22,RTCC Food Court,1,1,13.99,RTCC C and G MO,19176649,1,CG004,Trojan Acai Medium,C&G,C&G Meal Plan,False,0,15226204,15624038,Meal Exchange,Breakfast,RTCC_C&G
5,15624037,2026-04-09 10:03:21,RTCC Food Court,1,1,11.99,RTCC C and G MO,19176647,1,CG011,Protein Power Medium,C&G,C&G Meal Plan,False,0,15226203,15624037,Meal Exchange,Breakfast,RTCC_C&G
6,15624033,2026-04-09 10:03:20,RTCC Food Court,1,1,13.99,RTCC C and G MO,19176643,1,CG004,Trojan Acai Medium,C&G,C&G Meal Plan,False,0,15226199,15624033,Meal Exchange,Breakfast,RTCC_C&G
7,15624032,2026-04-09 10:03:18,RTCC Food Court,1,1,11.99,RTCC C and G MO,19176642,1,CG011,Protein Power Medium,C&G,C&G Meal Plan,False,0,15226198,15624032,Meal Exchange,Breakfast,RTCC_C&G
8,15624028,2026-04-09 10:03:16,RTCC Food Court,1,1,13.99,RTCC C and G MO,19176638,1,CG004,Trojan Acai Medium,C&G,C&G Meal Plan,False,0,15226194,15624028,Meal Exchange,Breakfast,RTCC_C&G
9,15623992,2026-04-09 10:01:28,RTCC Food Court,1,1,16.98,RTCC C and G MO,19176593,1,6788,Create Your Own,C&G,C&G Acai,False,0,15226158,15623992,Dining Dollars,Breakfast,RTCC_C&G


In [10]:
grouped = dff.groupby(['Unit', 'Meal Period'], observed=False)[['item_qty', 'item_price']].sum()

In [16]:
grouped = dff.groupby(['Unit', 'Meal Period'], observed=False)[['item_qty', 'item_price']].sum().unstack()

# 4. Add "Grand Totals" for Rows (Across Meal Periods)
grouped[('Total', 'Sum of item_qty')] = grouped['item_qty'].sum(axis=1)
grouped[('Total', 'Sum of item_price')] = grouped['item_price'].sum(axis=1)

# 5. Add "Grand Totals" for Columns (Across all Units)
# We create a final row by summing the columns
total_row = grouped.sum()
total_row.name = 'Grand Total'
final_df = pd.concat([grouped, total_row.to_frame().T])

# 6. Formatting the Headers
# Currently, columns are (item_qty, Breakfast), (item_qty, Lunch), etc.
# We swap levels to get (Meal Period, Measure) to match your requirement
final_df = final_df.swaplevel(0, 1, axis=1).sort_index(axis=1, level=0)

In [17]:
final_df

Meal Period  Breakfast          Sum of item_price Sum of item_qty
            item_price item_qty             Total           Total
RTCC_C&G        134.89     10.0            134.89            10.0
Grand Total     134.89     10.0            134.89            10.0

In [23]:
final_df.index

Index(['RTCC_C&G', 'Grand Total'], dtype='object')

In [22]:
final_df[('Breakfast', 'item_price')]

RTCC_C&G       134.89
Grand Total    134.89
Name: (Breakfast, item_price), dtype: float64

In [24]:
copy_df = final_df.copy()

In [27]:
# 1. Define the exact columns you want
target_columns = [
    ('Units', ''), 
    ('Breakfast', 'Items Sold'), ('Breakfast', 'Sales'),
    ('Lunch', 'Items Sold'), ('Lunch', 'Sales'),
    ('Dinner', 'Items Sold'), ('Dinner', 'Sales'),
    ('Total Items Sold', ''),
    ('Total Sales', '')
]
multi_col = pd.MultiIndex.from_tuples(target_columns)

# 2. Extract data safely from final_df
# We use .get() so if a column is missing (like Lunch), it doesn't crash
units = final_df.index.values
brk_qty = final_df.get(('Breakfast', 'item_qty'), 0)
brk_price = final_df.get(('Breakfast', 'item_price'), 0)
total_qty = final_df.get(('Sum of item_qty', 'Total'), 0)
total_sales = final_df.get(('Sum of item_price', 'Total'), 0)

# 3. Create a dictionary mapping the NEW columns to the data
data_map = {
    ('Units', ''): units,
    ('Breakfast', 'Items Sold'): brk_qty,
    ('Breakfast', 'Sales'): brk_price,
    ('Lunch', 'Items Sold'): 0,  # Placeholder for future data
    ('Lunch', 'Sales'): 0,        # Placeholder for future data
    ('Dinner', 'Items Sold'): 0, # Placeholder for future data
    ('Dinner', 'Sales'): 0,       # Placeholder for future data
    ('Total Items Sold', ''): total_qty,
    ('Total Sales', ''): total_sales
}

# 4. Build the final report
report_df = pd.DataFrame(data_map, columns=multi_col)

In [28]:
report_df

Units  Breakfast              Lunch           Dinner        \
                         Items Sold   Sales Items Sold Sales Items Sold Sales   
RTCC_C&G        RTCC_C&G       10.0  134.89          0     0          0     0   
Grand Total  Grand Total       10.0  134.89          0     0          0     0   

            Total Items Sold Total Sales  
                                          
RTCC_C&G                10.0      134.89  
Grand Total             10.0      134.89

In [30]:
from excel_formats import (
    dict_header_format,
    dict_number_format,
    dict_currency_format,
    dict_percent_format,
    dict_index_format,
    dict_totals_index_format,
    dict_total_currency_format,
    dict_total_percent_format,
    dict_merge_format,
    dict_total_number_format
)

In [33]:
import pandas as pd
from io import BytesIO

def export_report_to_excel(df, month, year):
    output = BytesIO()
    date_str = f"{month} {year}"
    
    with pd.ExcelWriter(output, engine='xlsxwriter') as writer:
        workbook = writer.book
        sheet = workbook.add_worksheet('Sales Report')
        
        # 1. Define Formats
        # Header 1 (Top): No bottom border
        dict_header0 = dict_header_format.copy()
        dict_header0.update({'bottom': 0}) 
        
        # Header 2 (Bottom): Keep the thick bottom border (6)
        dict_header1 = dict_header_format.copy()
        
        fmts = {k: workbook.add_format(v) for k, v in {
            'header0': dict_header0,
            'header1': dict_header1,
            'num': dict_number_format,
            'curr': dict_currency_format,
            'unit': dict_index_format,
            'total_unit': dict_totals_index_format,
            'total_num': dict_total_number_format,
            'total_curr': dict_total_currency_format,
            'title': dict_merge_format
        }.items()}

        # 2. Setup Offsets (Start at Column B, Row 3)
        start_row = 2  # Row 3
        start_col = 1  # Column B

        # 3. Write Title (Merged across the table width)
        sheet.merge_range(start_row - 1, start_col, start_row - 1, start_col + 8, 
                          f'USC Retail Sales Report - {date_str}', fmts['title'])

        # 4. Write Headers with Merging
        # We manually define the merge ranges for the top level
        # Units, Total Items, and Total Sales are vertically merged
        sheet.merge_range(start_row, start_col, start_row + 1, start_col, 'Units', fmts['header1'])
        
        # Meal Periods: Merged horizontally across 2 columns
        sheet.merge_range(start_row, start_col + 1, start_row, start_col + 2, 'Breakfast', fmts['header0'])
        sheet.merge_range(start_row, start_col + 3, start_row, start_col + 4, 'Lunch', fmts['header0'])
        sheet.merge_range(start_row, start_col + 5, start_row, start_col + 6, 'Dinner', fmts['header0'])
        
        # Totals: Vertically merged
        sheet.merge_range(start_row, start_col + 7, start_row + 1, start_col + 7, 'Total Items Sold', fmts['header1'])
        sheet.merge_range(start_row, start_col + 8, start_row + 1, start_col + 8, 'Total Sales', fmts['header1'])

        # Write Second Level (Sub-headers)
        sub_headers = ['Items Sold', 'Sales', 'Items Sold', 'Sales', 'Items Sold', 'Sales']
        for i, text in enumerate(sub_headers):
            sheet.write(start_row + 1, start_col + 1 + i, text, fmts['header1'])

        # 5. Write Data
        for i, row in enumerate(df.values):
            curr_row = start_row + 2 + i
            is_grand_total = "Grand Total" in str(row)
            
            for j, value in enumerate(row):
                # Check if column should be currency
                is_currency = "Sales" in df.columns[j] or "Sales" in df.columns[j]
                
                if is_grand_total:
                    fmt = fmts['total_curr'] if is_currency else fmts['total_num']
                    if j == 0: fmt = fmts['total_unit']
                else:
                    fmt = fmts['curr'] if is_currency else fmts['num']
                    if j == 0: fmt = fmts['unit']
                
                sheet.write(curr_row, start_col + j, value, fmt)

        # 6. Set Column Widths (Adjusting for the offset)
        sheet.set_column(start_col, start_col, 25) 
        sheet.set_column(start_col + 1, start_col + 8, 15)

    output.seek(0)
    return output

In [34]:
# report_df is the dataframe we built in the previous step
excel_file = export_report_to_excel(report_df, "April", "2026")

with open("Final_Report.xlsx", "wb") as f:
    f.write(excel_file.getbuffer())